# Python fundamentals for the FANUC starter

**Time:** 60–90 minutes  
**Prerequisites:** Notebook 00 (or basic Git). Python 3.12+ recommended.

## Learning objectives

- Work with numbers, strings, lists, dicts, and loops
- Import project modules and read configuration data
- Load JSON and handle errors cleanly
- Write small functions that match hackathon code style

## Checkpoint

You can load `scene_config` and print cube colors and stack order without running Isaac Sim.

In [ ]:
from pathlib import Path
import sys

def repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "fanuc_er4ia_sim" / "scene_config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find fanuc_er4ia_sim")

ROOT = repo_root()
SIM = ROOT / "fanuc_er4ia_sim"
if str(SIM) not in sys.path:
    sys.path.insert(0, str(SIM))
print("Using simulation package:", SIM)

## Values and collections

In [ ]:
import scene_config as C

edge_mm = C.CUBE_EDGE * 1000
print(f"Cube edge: {edge_mm:.1f} mm")
print("Stack colors (bottom → top):", " → ".join(C.CUBE_STACK_COLORS))
print("Number of cubes on table:", len(C.SORT_CUBES))

colors = [color for _, _, color in C.SORT_CUBES]
from collections import Counter

counts = Counter(colors)
print("Color counts on table:", dict(counts))

**TODO:** Assert that each of yellow, blue, and red appears exactly twice on the table (hackathon layout).

In [ ]:
assert counts["yellow"] == counts["blue"] == counts["red"] == 2
print("Layout color check passed.")

## Loops and functions

In [ ]:
import math

def table_positions():
    # Return list of (x, y, color) for each sort cube.
    return [(x, y, color) for x, y, color in C.SORT_CUBES]

def min_separation(points):
    # Minimum XY distance between distinct cube centres.
    best = float("inf")
    for i, (x1, y1, _) in enumerate(points):
        for x2, y2, _ in points[i + 1 :]:
            best = min(best, math.dist((x1, y1), (x2, y2)))
    return best

pts = table_positions()
print(f"Minimum centre separation: {min_separation(pts):.3f} m")

## JSON and paths (pipeline artifacts)

In [ ]:
import json

# Example structure written by plan_sort.py (field names only — file may not exist yet)
example_plan = {
    "task": "cube_stack",
    "picks": [],
    "observations": [],
}
print(json.dumps(example_plan, indent=2)[:200], "...")

runs = SIM / "runs"
print("Runs directory exists:", runs.is_dir())
if runs.is_dir():
    sub = sorted(runs.iterdir())[:3]
    print("Sample run folders:", [p.name for p in sub if p.is_dir()])

## Exceptions

In [ ]:
def load_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError:
        return None
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON in {path}") from exc

missing = load_json(SIM / "runs" / "does_not_exist" / "sort_plan.json")
print("Missing file returns:", missing)

## Joint limits (dict practice)

Robot joint limits live in `JOINT_LIMITS_DEG`. 

**TODO:** Print the joint with the largest allowable range in degrees.

In [ ]:
limits = C.JOINT_LIMITS_DEG
span = {name: hi - lo for name, (lo, hi) in limits.items()}
widest = max(span, key=span.get)
print(f"Largest range: {widest} ({span[widest]:.1f}°)")

## Troubleshooting

| Error | Likely cause |
|-------|----------------|
| `ModuleNotFoundError: scene_config` | `sys.path` does not include `fanuc_er4ia_sim` — run the setup cell |
| Wrong cube counts | You edited `scene_config.py`; re-read defaults |

## Cleanup

No servers to stop. If you created scratch files for exercises, delete or gitignore them.